In [10]:
import pandas as pd
import numpy as np

In [11]:
# Load the CSV file into a DataFrame
df_sen = pd.read_csv("new_sentence_embeddings.csv")

# Show the first few rows of the dataframe to verify loading
df_sen.head()

,filename,Sentence,processed_sentences,embedding
0,State v. Allen.txt,Donovan Allen challenges his aggravated first-...,donovan allen challenge aggravated firstdegree...,"[0.05029255151748657, 0.012195201590657234, -0..."
1,State v. Allen.txt,He argues that there was insufficient evidence...,argues insufficient evidence show murder preme...,"[0.034305389970541, 0.00682568596675992, -0.03..."
2,State v. Allen.txt,Allen also claims that the trial court made nu...,allen also claim trial court made numerous evi...,"[0.002080693142488599, 0.009060398675501347, 0..."
3,State v. Allen.txt,"Additionally, he alleges violations of his rig...",additionally alleges violation right counsel p...,"[0.026385504752397537, 0.011483738198876381, 0..."
4,State v. Allen.txt,"The Court of Appeal affirms the conviction, fi...",court appeal affirms conviction finding error ...,"[0.025612568482756615, 0.03851771727204323, -0..."


In [33]:
import ast
# Convert each string in 'embedding' column to a NumPy array
df_sen['embedding'] = df_sen['embedding'].apply(lambda x: np.array(ast.literal_eval(x)))

### Use Convex

In [12]:
from functions import *

def testing_convex(X, y, num_topics, lam, max_iter, random_state):
    # Define train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

    # Semi NMF initialization
    F, G = semiNMF(X_train, num_topics, max_iter)
    
    # Convex NMF
    print('\nConvex NMF:')
    G_train, W = ConvexNMF(X_train, G.T, max_iter)
    XTX = X_train.T @ X_train
    G_test = (G_train.T @ np.linalg.inv(X_train @ X_train.T) @ X_train @ X_test.T).T
    #(np.linalg.pinv(W) @ X_train @ np.linalg.inv(XTX) @ X_test.T).T
    print('Grid search:')
    grid_search = grid_search_svm(G_train, y_train, G_test, y_test)
    print('Best model:')
    convex_acc = best_svm(grid_search, G_train, G_test, y_train, y_test)
    
    return convex_acc, W

In [13]:
df = pd.read_csv('matrixRecommendationJusticeProcessed_df.csv', header=None)
df

,0,1,2,3,4,5,6,7,8,9,...,1526,1527,1528,1529,1530,1531,1532,1533,1534,1535
0,0.001805,-0.017398,-0.005838,-0.025201,-0.027487,0.032880,-0.002170,-0.016509,-0.035098,-0.025324,...,-0.006194,-0.009171,0.022710,-0.021053,-0.010267,0.001814,-0.005287,-0.000740,-0.027446,-0.001014
1,0.015217,0.007216,-0.001332,-0.043284,-0.024253,0.005657,0.002244,-0.005640,-0.019965,-0.030921,...,0.000235,-0.002049,0.031787,-0.029731,0.002075,-0.007838,0.004491,-0.015420,-0.022372,0.020127
2,0.004047,0.015472,0.001917,-0.033242,-0.033026,0.021729,0.013135,-0.027026,-0.023621,-0.010966,...,0.006878,-0.016094,0.006392,-0.015959,0.007898,0.000601,0.022918,-0.010135,-0.026634,-0.006189
3,-0.005267,-0.001192,0.015698,-0.031528,-0.026013,0.012542,0.012874,-0.026304,-0.028611,-0.036036,...,-0.001695,0.015300,0.016560,-0.042374,0.005837,-0.001359,-0.027047,-0.004253,-0.039085,-0.002211
4,0.006276,0.012211,0.010281,-0.027250,-0.026727,0.008491,0.008451,-0.006518,-0.031847,-0.022652,...,-0.002051,-0.019516,0.013464,-0.008565,-0.013926,0.007064,0.014865,-0.022250,0.002284,0.006782
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,0.004164,0.003739,-0.000084,-0.048068,-0.018115,0.028984,0.002493,-0.019018,-0.038964,-0.009429,...,-0.003450,-0.032090,0.028639,-0.018872,-0.010988,0.014744,0.008653,-0.023463,-0.013483,-0.012044
136,0.008245,0.009895,0.006849,-0.023689,-0.040125,0.023214,0.011838,-0.021855,-0.053164,-0.014914,...,-0.009298,-0.006574,0.022915,-0.025631,0.015770,0.004948,0.005566,0.003027,-0.025658,-0.017930
137,0.004611,-0.020028,0.003694,-0.031843,-0.016686,0.036236,0.000262,-0.015280,-0.042130,-0.045432,...,-0.010689,-0.002372,0.022266,-0.025308,-0.008159,0.005611,-0.000901,0.001431,-0.031652,0.000110
138,0.012210,0.001298,-0.016831,-0.043824,-0.035902,0.020018,0.006438,-0.022085,-0.028167,-0.023832,...,0.007022,-0.017684,0.010049,-0.015377,-0.002354,-0.015630,-0.001004,-0.012683,-0.028967,-0.007482


In [14]:
X = df.values
Y = np.vstack((np.tile([1, 0], (70, 1)), np.tile([0, 1], (70, 1))))
y = Y[:, 1] # class 0: exonerated

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
Y_one_hot = np.zeros((y.size, y.max() + 1))
Y_one_hot[np.arange(y.size), y] = 1
Y = Y_one_hot
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
#X_train, X_test = X_train.T, X_test.T

F, G = semiNMF(X_train, 10, 1000)

In [42]:
num_topics = 15
lam = 1
max_iter = 100
random_state = 0

convex_acc = testing_convex(X, y, num_topics, lam, max_iter, random_state)


Convex NMF:
Grid search:
Best Parameters: {'C': 1000, 'gamma': 1, 'kernel': 'rbf'}
Best Cross-Validation Score: 0.6162055335968379
Test Accuracy: 0.7857142857142857
Best model:
Train Accuracy: 0.7589285714285714
Test Accuracy: 0.7857142857142857

Actual vs Predicted labels:
[0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1]
[0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1]
Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.86      0.80        14
           1       0.83      0.71      0.77        14

    accuracy                           0.79        28
   macro avg       0.79      0.79      0.78        28
weighted avg       0.79      0.79      0.78        28



In [53]:
convex_acc_ls = []
for i in range(20):
    convex_acc = testing_convex(X, y, num_topics, lam, max_iter, i)
    convex_acc_ls.append(convex_acc)

convex_acc_ls


Convex NMF:
Grid search:
Best Parameters: {'C': 100, 'gamma': 1, 'kernel': 'rbf'}
Best Cross-Validation Score: 0.6430830039525691
Test Accuracy: 0.7857142857142857
Best model:
Train Accuracy: 0.6785714285714286
Test Accuracy: 0.7857142857142857

Convex NMF:
Grid search:
Best Parameters: {'C': 1000, 'gamma': 1, 'kernel': 'rbf'}
Best Cross-Validation Score: 0.5802371541501976
Test Accuracy: 0.7142857142857143
Best model:
Train Accuracy: 0.6696428571428571
Test Accuracy: 0.7142857142857143

Convex NMF:
Grid search:
Best Parameters: {'C': 100, 'gamma': 1, 'kernel': 'linear'}
Best Cross-Validation Score: 0.5810276679841897
Test Accuracy: 0.6428571428571429
Best model:
Train Accuracy: 0.5892857142857143
Test Accuracy: 0.6428571428571429

Convex NMF:
Grid search:
Best Parameters: {'C': 1000, 'gamma': 1, 'kernel': 'linear'}
Best Cross-Validation Score: 0.5818181818181818
Test Accuracy: 0.7857142857142857
Best model:
Train Accuracy: 0.6517857142857143
Test Accuracy: 0.7857142857142857

Convex 

KeyboardInterrupt: 

### Use Highest Accuracy Model

In [43]:
# modify best_svm function to show the predictions
from sklearn.metrics import accuracy_score, classification_report

def best_svm(grid_search, X_train, X_test, y_train, y_test):
    """
    Parameters:
    grid_search: grid search returned by function grid_search_svm
    - X_train (array-like): Training features.
    - y_train (array-like): Training labels.
    - X_test (array-like): Testing features.
    - y_test (array-like): Testing labels.
    
    Return: test accuracy of the best SVM model, actual labels, and predicted labels
    """
    svm = grid_search.best_estimator_
    svm.fit(X_train, y_train)
    
    # Predict on the test set
    y_pred = svm.predict(X_test)
    
    # Calculate accuracy
    test_accuracy = accuracy_score(y_test, y_pred)
    y_train_pred = svm.predict(X_train)
    train_accuracy = accuracy_score(y_train, y_train_pred)
    
    # Print the classification report
    report = classification_report(y_test, y_pred)
    
    print(f'Train Accuracy: {train_accuracy}')
    print(f'Test Accuracy: {test_accuracy}')
    
    # Collect actual and predicted labels into lists
    actual_labels = y_test.tolist()  # Convert to list
    predicted_labels = y_pred.tolist()  # Convert to list
    
    # Print actual vs predicted labels
    print("\nActual vs Predicted labels:")
    print(actual_labels)
    print(predicted_labels)
    
    # Optionally, print the classification report
    print(f"Classification Report:\n{report}")
    
    return test_accuracy, actual_labels, predicted_labels

In [44]:
num_topics = 15
lam = 1
max_iter = 100
random_state = 0

convex_acc, W = testing_convex(X, y, num_topics, lam, max_iter, random_state)


Convex NMF:
Grid search:
Best Parameters: {'C': 1000, 'gamma': 1, 'kernel': 'rbf'}
Best Cross-Validation Score: 0.6162055335968379
Test Accuracy: 0.7857142857142857
Best model:
Train Accuracy: 0.7589285714285714
Test Accuracy: 0.7857142857142857

Actual vs Predicted labels:
[0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1]
[0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1]
Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.86      0.80        14
           1       0.83      0.71      0.77        14

    accuracy                           0.79        28
   macro avg       0.79      0.79      0.78        28
weighted avg       0.79      0.79      0.78        28



In [45]:
# Split the data
train_indices, test_indices = train_test_split(range(140), test_size=0.2, random_state=0)

print("Train Indices:", train_indices)
print("Test Indices:", test_indices)

Train Indices: [18, 10, 56, 43, 109, 48, 107, 83, 60, 106, 89, 78, 44, 30, 62, 121, 114, 73, 95, 84, 134, 13, 54, 94, 120, 118, 15, 68, 40, 61, 86, 66, 3, 52, 112, 116, 6, 124, 12, 85, 136, 127, 126, 11, 93, 98, 41, 100, 1, 96, 129, 42, 4, 113, 17, 38, 5, 53, 133, 108, 0, 34, 28, 55, 75, 35, 23, 74, 31, 101, 57, 119, 65, 32, 128, 14, 105, 19, 29, 49, 125, 99, 82, 64, 139, 79, 69, 138, 80, 115, 20, 135, 72, 77, 25, 37, 81, 130, 46, 132, 39, 58, 88, 70, 87, 36, 21, 9, 103, 67, 117, 47]
Test Indices: [45, 59, 7, 50, 92, 27, 131, 137, 122, 8, 111, 16, 63, 76, 123, 97, 104, 110, 33, 91, 90, 22, 102, 24, 2, 51, 26, 71]


In [46]:
W.shape #112 because of training size

(112, 15)

In [47]:
X_train.shape

(112, 1536)

## Compare sentence embeddings with topic embeddings

In [48]:
# dimension of A (or XW for convex): 1536 by 15 (topic embeddings)

XW = (X_train.T @ W).T
XW.shape

(15, 1536)

In [49]:
from sklearn.metrics.pairwise import cosine_similarity

# Assuming df_merged['sentence_embedding'] contains list-like embeddings
# Convert it to a matrix form for efficient computation
sentence_embeddings = np.vstack(df_sen['embedding'].values)

# Define a list to hold the results
results = []

# Iterate over each topic embedding in XW (assume XW has shape (15, 1536))
for topic_idx, topic_embedding in enumerate(XW):
    # Compute cosine similarity between the topic and all sentence embeddings
    similarities = cosine_similarity(sentence_embeddings, topic_embedding.reshape(1, -1)).flatten()
    
    # Get the indices of the top 5 closest sentence embeddings
    top_5_indices = np.argsort(similarities)[-5:][::-1]  # Sort and get last 5 in descending order

    # Save the corresponding rows from df_merged and the topic index
    for idx in top_5_indices:
        row = df_merged.iloc[idx].copy()  # Copy the row from df_merged
        row['closest_topic'] = topic_idx  # Add the topic this sentence is closest to
        results.append(row)

# Create a new dataframe from the results
df_closest_sentences = pd.DataFrame(results)

In [50]:
# To display all rows
pd.set_option('display.max_rows', None)

# To display all columns
pd.set_option('display.max_columns', None)

df_closest_sentences

filename  \
80   State v. Engesser.txt   
233     State v. Pease.txt   
747   State v. Bescher.txt   
384     People v Colon.txt   
982  People v. Coleman.txt   
80   State v. Engesser.txt   
233     State v. Pease.txt   
747   State v. Bescher.txt   
384     People v Colon.txt   
982  People v. Coleman.txt   
80   State v. Engesser.txt   
233     State v. Pease.txt   
747   State v. Bescher.txt   
384     People v Colon.txt   
982  People v. Coleman.txt   
80   State v. Engesser.txt   
233     State v. Pease.txt   
747   State v. Bescher.txt   
384     People v Colon.txt   
982  People v. Coleman.txt   
80   State v. Engesser.txt   
747   State v. Bescher.txt   
233     State v. Pease.txt   
384     People v Colon.txt   
982  People v. Coleman.txt   
80   State v. Engesser.txt   
233     State v. Pease.txt   
747   State v. Bescher.txt   
384     People v Colon.txt   
982  People v. Coleman.txt   
80   State v. Engesser.txt   
233     State v. Pease.txt   
747   State v. Bescher.txt   
384     People v Colon.txt   
982  People v. Coleman.txt   
80   State v. Engesser.txt   
233     State v. Pease.txt   
747   State v. Bescher.txt   
384     People v Colon.txt   
982  People v. Coleman.txt   
80   State v. Engesser.txt   
233     State v. Pease.txt   
747   State v. Bescher.txt   
384     People v Colon.txt   
982  People v. Coleman.txt   
80   State v. Engesser.txt   
233     State v. Pease.txt   
747   State v. Bescher.txt   
982  People v. Coleman.txt   
384     People v Colon.txt   
80   State v. Engesser.txt   
233     State v. Pease.txt   
747   State v. Bescher.txt   
384     People v Colon.txt   
982  People v. Coleman.txt   
80   State v. Engesser.txt   
747   State v. Bescher.txt   
233     State v. Pease.txt   
384     People v Colon.txt   
982  People v. Coleman.txt   
80   State v. Engesser.txt   
233     State v. Pease.txt   
747   State v. Bescher.txt   
384     People v Colon.txt   
982  People v. Coleman.txt   
80   State v. Engesser.txt   
233     State v. Pease.txt   
747   State v. Bescher.txt   
384     People v Colon.txt   
982  People v. Coleman.txt   
80   State v. Engesser.txt   
747   State v. Bescher.txt   
233     State v. Pease.txt   
384     People v Colon.txt   
982  People v. Coleman.txt   

                                                                                                                                                                                                                                                                          Sentence  \
80   The defendant argues that the trial court made errors by denying the suppression of blood draw results, allowing a police officer to testify about his perception of the defendant's truthfulness, refusing hearsay evidence, and failing to instruct the jury on spoliation.   
233                                                                  The Superior Court had granted Pea a new trial, but the Supreme Court reversed the ruling of the Court of Appeals and remanded the case for reinstatement of the Superior Court's order granting a new trial.   
747                                                                                                                                                             Bescher was also charged with additional criminal actions, but those charges were dismissed during jury selection.   
384                                                                                                                                                                                         The appellate court, in an unpublished opinion, reversed and remanded for a new trial.   
982                                                                                                                                                                              The circuit court initially dismissed the claim but allowed an amendment to part of the petition.   
80   The defendant argues that the trial court made errors by denying

In [51]:
df_topics = df_closest_sentences.copy()
df_topics = df_topics.drop(columns=['embedding','vector'])
df_topics

,filename,Sentence,processed_sentences,closest_topic
80,State v. Engesser.txt,"The defendant argues that the trial court made errors by denying the suppression of blood draw results, allowing a police officer to testify about his perception of the defendant's truthfulness, refusing hearsay evidence, and failing to instruct the jury on spoliation.",defendant argues trial court made error denying suppression blood draw result allowing police officer testify perception defendant truthfulness refusing hearsay evidence failing instruct jury spoliation,0
233,State v. Pease.txt,"The Superior Court had granted Pea a new trial, but the Supreme Court reversed the ruling of the Court of Appeals and remanded the case for reinstatement of the Superior Court's order granting a new trial.",superior court granted pea new trial supreme court reversed ruling court appeal remanded case reinstatement superior court order granting new trial,0
747,State v. Bescher.txt,"Bescher was also charged with additional criminal actions, but those charges were dismissed during jury selection.",bescher also charged additional criminal action charge dismissed jury selection,0
384,People v Colon.txt,"The appellate court, in an unpublished opinion, reversed and remanded for a new trial.",appellate court unpublished opinion reversed remanded new trial,0
982,People v. Coleman.txt,The circuit court initially dismissed the claim but allowed an amendment to part of the petition.,circuit court initially dismissed claim allowed amendment part petition,0
80,State v. Engesser.txt,"The defendant argues that the trial court made errors by denying the suppression of blood draw results, allowing a police officer to testify about his perception of the defendant's truthfulness, refusing hearsay evidence, and failing to instruct the jury on spoliation.",defendant argues trial court made error denying suppression blood draw result allowing police officer testify perception defendant truthfulness refusing hearsay evidence failing instruct jury spoliation,1
233,State v. Pease.txt,"The Superior Court had granted Pea a new trial, but the Supreme Court reversed the ruling of the Court of Appeals and remanded the case for reinstatement of the Superior Court's order granting a new trial.",superior court granted pea new trial supreme court reversed ruling court appeal remanded case reinstatement superior court order granting new trial,1
747,State v. Bescher.txt,"Bescher was also charged with additional criminal actions, but those charges were dismissed during jury selection.",bescher also charged additional criminal action charge dismissed jury selection,1
384,People v Colon.txt,"The appellate court, in an unpublished opinion, reversed and remanded for a new trial.",appellate court unpublished opinion reversed remanded new trial,1
982,People v. Coleman.txt,The circuit court initially dismissed the claim but allowed an amendment to part of the petition.,circuit court initially dismissed claim allowed amendment part petition,1


In [52]:
# Set pandas options to display all columns and avoid truncating their content
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.max_colwidth', None)  # Show full content of each column

df_topics

,filename,Sentence,processed_sentences,closest_topic
80,State v. Engesser.txt,"The defendant argues that the trial court made errors by denying the suppression of blood draw results, allowing a police officer to testify about his perception of the defendant's truthfulness, refusing hearsay evidence, and failing to instruct the jury on spoliation.",defendant argues trial court made error denying suppression blood draw result allowing police officer testify perception defendant truthfulness refusing hearsay evidence failing instruct jury spoliation,0
233,State v. Pease.txt,"The Superior Court had granted Pea a new trial, but the Supreme Court reversed the ruling of the Court of Appeals and remanded the case for reinstatement of the Superior Court's order granting a new trial.",superior court granted pea new trial supreme court reversed ruling court appeal remanded case reinstatement superior court order granting new trial,0
747,State v. Bescher.txt,"Bescher was also charged with additional criminal actions, but those charges were dismissed during jury selection.",bescher also charged additional criminal action charge dismissed jury selection,0
384,People v Colon.txt,"The appellate court, in an unpublished opinion, reversed and remanded for a new trial.",appellate court unpublished opinion reversed remanded new trial,0
982,People v. Coleman.txt,The circuit court initially dismissed the claim but allowed an amendment to part of the petition.,circuit court initially dismissed claim allowed amendment part petition,0
80,State v. Engesser.txt,"The defendant argues that the trial court made errors by denying the suppression of blood draw results, allowing a police officer to testify about his perception of the defendant's truthfulness, refusing hearsay evidence, and failing to instruct the jury on spoliation.",defendant argues trial court made error denying suppression blood draw result allowing police officer testify perception defendant truthfulness refusing hearsay evidence failing instruct jury spoliation,1
233,State v. Pease.txt,"The Superior Court had granted Pea a new trial, but the Supreme Court reversed the ruling of the Court of Appeals and remanded the case for reinstatement of the Superior Court's order granting a new trial.",superior court granted pea new trial supreme court reversed ruling court appeal remanded case reinstatement superior court order granting new trial,1
747,State v. Bescher.txt,"Bescher was also charged with additional criminal actions, but those charges were dismissed during jury selection.",bescher also charged additional criminal action charge dismissed jury selection,1
384,People v Colon.txt,"The appellate court, in an unpublished opinion, reversed and remanded for a new trial.",appellate court unpublished opinion reversed remanded new trial,1
982,People v. Coleman.txt,The circuit court initially dismissed the claim but allowed an amendment to part of the petition.,circuit court initially dismissed claim allowed amendment part petition,1


In [53]:
output_file_path = "processed_df_topics_15.csv"
df_topics.to_csv(output_file_path, index=False)

print(f"Saved df_topics to {output_file_path}")

Saved df_topics to processed_df_topics_15.csv


In [41]:
XW

array([[ 0.0228219 ,  0.00566474,  0.01159775, ..., -0.06303771,
        -0.08202817,  0.01027391],
       [ 0.02332084,  0.00458044,  0.01030967, ..., -0.05701624,
        -0.07702285,  0.00421468],
       [ 0.02229532,  0.00842431,  0.01157302, ..., -0.0575693 ,
        -0.07987196,  0.00459096],
       ...,
       [ 0.02449313,  0.00488587,  0.01615931, ..., -0.06079361,
        -0.08541765,  0.00670586],
       [ 0.01920373,  0.00760887,  0.00983017, ..., -0.05830106,
        -0.07759007,  0.01268521],
       [ 0.02767566,  0.0057366 ,  0.01083242, ..., -0.05672895,
        -0.0800297 ,  0.00792358]])